In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow.keras.layers as tfl
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing import image, image_dataset_from_directory

In [2]:
BATCH_SIZE = 8
IMG_SIZE = (224,224)
directory = "testimg"
IMG_SHAPE = IMG_SIZE + (3,)
directory = "testimg"
train_dataset = image_dataset_from_directory(directory,
                                             shuffle=True,
                                             batch_size=BATCH_SIZE,
                                             image_size=IMG_SIZE,
                                             validation_split=0.2,
                                             subset='training',
                                             seed=42)

validation_dataset = image_dataset_from_directory(directory,
                                                  shuffle=True,
                                                  batch_size=BATCH_SIZE,
                                                  image_size=IMG_SIZE,
                                                  validation_split=0.2,
                                                  subset='validation',
                                                  seed=42)
image_count = tf.data.experimental.cardinality(train_dataset).numpy()
print(f"Train batches: {image_count}, Total images: {image_count * BATCH_SIZE}")

NotFoundError: Could not find directory testimg

In [ ]:
class_names = train_dataset.class_names
print(class_names)

plt.figure(figsize=(10, 5))
for images, labels in train_dataset.take(1):
    for i in range(8):
        ax = plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")

In [ ]:
AUTOTUNE = tf.data.experimental.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
preprocess_input = tf.keras.applications.mobilenet.preprocess_input

def data_augumenter():
    with tf.device('/cpu:0'):
        data_augmentation = tf.keras.Sequential([])
        data_augmentation.add(tf.keras.layers.RandomFlip('horizontal'))
        data_augmentation.add(tf.keras.layers.RandomRotation(0.2))
        data_augmentation.add(tf.keras.layers.RandomContrast(0.2))
        data_augmentation.add(tf.keras.layers.RandomBrightness(0.2))
    return data_augmentation

data_augmentation = data_augumenter()

In [ ]:
for image, _ in train_dataset.take(1):
    plt.figure(figsize=(10, 5))
    first_image = image[1]
    for i in range(8):
        ax = plt.subplot(2, 4, i + 1)
        augmented_image = data_augmentation(tf.expand_dims(first_image, 0))
        plt.imshow(augmented_image[0] / 255)
        plt.axis('off')

In [ ]:
from tensorflow.keras.applications import MobileNetV2

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

In [ ]:
model = tf.keras.Sequential([
    data_augmentation,
    layers.Rescaling(1./255),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(len(class_names), activation='softmax')
])

In [ ]:
Opt = optimizers.Adam(learning_rate=0.0001)
model.compile(optimizer=Opt,
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])
# Train the model
history = model.fit(train_dataset, epochs=10,
    validation_data=(validation_dataset))

In [ ]:
model.evaluate(train_dataset)
model.save('MLproject/shapedet.keras')
print("saved")

In [ ]:
import cv2
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.image import img_to_array
import matplotlib.pyplot as plt

def hex_to_hsv(hex_code):
    hex_code = hex_code.lstrip('#')
    rgb = tuple(int(hex_code[i:i+2], 16) for i in (0, 2, 4))
    bgr = np.uint8([[rgb[::-1]]])
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    return hsv[0][0]

df = pd.read_csv('MLproject/color_names.csv', usecols=[0,1])
df.columns = ['Name', 'Hex']
df['Hex'] = df['Hex'].str.strip().str.lstrip('#')

color_ranges = {}
tolerancija = 20

for _, row in df.iterrows():
    hsv = hex_to_hsv(row['Hex'])
    lower = np.clip(hsv - [tolerancija, 100, 100], 0, 255)
    upper = np.clip(hsv + [tolerancija, 255, 255], 0, 255)
    color_ranges[row['Name']] = (lower.astype(np.uint8), upper.astype(np.uint8), row['Hex'])

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
cv2.namedWindow("Live Feed", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Live Feed", 640, 480)
font = cv2.FONT_HERSHEY_SIMPLEX

while True:
    ret, frame = cap.read()
    if not ret:
        print("Camera failed")
        break

    cv2.imshow("Live Feed", frame)
    tipka = cv2.waitKey(1) & 0xFF

    if tipka == ord('c'): 
        print("Picture taken :)")
        original_frame = frame.copy()
        hsv_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

     
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        edges = cv2.Canny(gray, 50, 150)
        edges = cv2.dilate(edges, None, iterations=2)
        contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        for contour in contours:
            area = cv2.contourArea(contour)
            if area < 1000:
                continue

            x, y, w, h = cv2.boundingRect(contour)


            roi_hsv = hsv_frame[y:y+h, x:x+w]
            detections = []
            for name, (lower, upper, hex_code) in color_ranges.items():
                mask = cv2.inRange(roi_hsv, lower, upper)
                count = cv2.countNonZero(mask)
                if count > 300:
                    detections.append((name, hex_code, count))
            detections = sorted(detections, key=lambda x: x[2], reverse=True)
            color_label = detections[0][0] if detections else "Unknown"
            color_hex = detections[0][1] if detections else "#000000"
            bgr_color = tuple(int(color_hex[i:i+2], 16) for i in (0, 2, 4))[::-1]


            roi = original_frame[y:y+h, x:x+w]
            roi_resized = cv2.resize(roi, (224, 224))
            roi_array = img_to_array(roi_resized) / 255.0
            roi_array = np.expand_dims(roi_array, axis=0)
            predictions = model.predict(roi_array)
            predicted_index = np.argmax(predictions)
            shape_label = class_names[predicted_index]


            label = f"{shape_label} ({color_label})"
            cv2.rectangle(frame, (x, y), (x+w, y+h), bgr_color, 2)
            cv2.putText(frame, label, (x, y - 10), font, 0.5, bgr_color, 2, cv2.LINE_AA)

        cv2.imshow("Processed Frame", frame)

    if tipka == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
